In [ ]:
# Install yfinance if not already installed
!pip install yfinance -q

import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# --- STRATEGY PARAMETERS ---
LOOKBACK_BARS = 200
BIN_RESOLUTION = 90
STDEV_LENGTH = 25
TRACKING_BARS = 140    # How many hourly bars back to look for historical POCs (~20 days)
TOLERANCE_PCT = 0.01   # 1% wiggle room

# --- CUSTOM TICKER LIST ---
TICKER_STRING = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM
"""
tickers_to_scan = TICKER_STRING.split()

def analyze_historical_pocs(ticker, df):
    try:
        df['stdev'] = df['Close'].rolling(window=STDEV_LENGTH).std(ddof=0) * 2
        df['PriceRef'] = np.where(df['Close'] > df['Open'], 
                                  df['Low'] - df['stdev'], 
                                  df['High'] + df['stdev'])
        df = df.dropna()
        
        if len(df) < LOOKBACK_BARS + TRACKING_BARS + 2:
            return None

        historical_buy_pocs = []
        historical_sell_pocs = []

        start_idx = len(df) - TRACKING_BARS - 2
        end_idx = len(df) - 2 

        for i in range(start_idx, end_idx):
            window = df.iloc[i-LOOKBACK_BARS : i]
            curr_close = window['Close'].iloc[-1]
            
            min_p = window['PriceRef'].min()
            max_p = window['PriceRef'].max()
            if pd.isna(min_p) or min_p == max_p:
                continue
                
            p_step = (max_p - min_p) / BIN_RESOLUTION
            b_mids = np.array([min_p + (b + 0.5) * p_step for b in range(BIN_RESOLUTION)])
            
            vols = np.zeros(BIN_RESOLUTION)
            pref = window['PriceRef'].values
            vol = window['Volume'].values
            
            for b in range(BIN_RESOLUTION):
                if abs(curr_close - b_mids[b]) < (p_step * 2):
                    continue
                mask = np.abs(pref - b_mids[b]) <= p_step
                vols[b] = np.sum(vol[mask])
                
            buy_mask = b_mids <= curr_close
            if np.any(buy_mask):
                buy_vols = vols.copy()
                buy_vols[~buy_mask] = 0
                max_buy_idx = np.argmax(buy_vols)
                if buy_vols[max_buy_idx] > 0:
                    historical_buy_pocs.append({"price": b_mids[max_buy_idx], "bar_created": i})

            sell_mask = b_mids > curr_close
            if np.any(sell_mask):
                sell_vols = vols.copy()
                sell_vols[~sell_mask] = 0
                max_sell_idx = np.argmax(sell_vols)
                if sell_vols[max_sell_idx] > 0:
                    historical_sell_pocs.append({"price": b_mids[max_sell_idx], "bar_created": i})

        valid_buy_pocs = []
        for poc in historical_buy_pocs:
            mitigated = False
            for j in range(poc["bar_created"] + 1, len(df) - 2):
                if df['Close'].iloc[j] < poc["price"] * (1 - TOLERANCE_PCT):
                    mitigated = True
                    break
            if not mitigated:
                valid_buy_pocs.append(poc["price"])

        valid_sell_pocs = []
        for poc in historical_sell_pocs:
            mitigated = False
            for j in range(poc["bar_created"] + 1, len(df) - 2):
                if df['Close'].iloc[j] > poc["price"] * (1 + TOLERANCE_PCT):
                    mitigated = True
                    break
            if not mitigated:
                valid_sell_pocs.append(poc["price"])

        valid_buy_pocs = list(set(valid_buy_pocs))
        valid_sell_pocs = list(set(valid_sell_pocs))

        prev = df.iloc[-2]
        curr = df.iloc[-1]
        
        setup = None
        hit_price = 0

        bull_engulfing = (prev['Close'] < prev['Open']) and \
                         (curr['Close'] > curr['Open']) and \
                         (curr['Open'] <= prev['Close']) and \
                         (curr['Close'] >= prev['Open'])

        if bull_engulfing:
            for poc in valid_buy_pocs:
                touched = (curr['Low'] <= poc * (1 + TOLERANCE_PCT)) or (prev['Low'] <= poc * (1 + TOLERANCE_PCT))
                held = curr['Close'] >= poc * (1 - TOLERANCE_PCT)
                if touched and held:
                    setup = "SELL PUT"
                    hit_price = poc
                    break

        bear_engulfing = (prev['Close'] > prev['Open']) and \
                         (curr['Close'] < curr['Open']) and \
                         (curr['Open'] >= prev['Close']) and \
                         (curr['Close'] <= prev['Open'])

        if bear_engulfing:
            for poc in valid_sell_pocs:
                touched = (curr['High'] >= poc * (1 - TOLERANCE_PCT)) or (prev['High'] >= poc * (1 - TOLERANCE_PCT))
                held = curr['Close'] <= poc * (1 + TOLERANCE_PCT)
                if touched and held:
                    setup = "SELL CALL"
                    hit_price = poc
                    break

        if setup:
            return {
                "Ticker": ticker,
                "Setup": setup,
                "Current Price": round(curr['Close'], 2),
                "Historical POC Hit": round(hit_price, 2)
            }
        return None

    except Exception as e:
        return None

# --- MAIN EXECUTION ---
print(f"Downloading 1-Hour data for {len(tickers_to_scan)} symbols...")
data = yf.download(tickers_to_scan, period="730d", interval="1h", group_by="ticker", threads=True, progress=False)

print("Scanning for 1-Hour Liquidity Pro Map Options Setups...")
results = []
calls_to_sell = []
puts_to_sell = []

for ticker in tqdm(tickers_to_scan):
    try:
        if len(tickers_to_scan) == 1:
            df = data.copy()
        else:
            df = data[ticker].dropna()
    except KeyError:
        continue
        
    if df.empty:
        continue
        
    result = analyze_historical_pocs(ticker, df)
    if result:
        results.append(result)
        # Separate the lists
        if result["Setup"] == "SELL CALL":
            calls_to_sell.append(ticker)
        elif result["Setup"] == "SELL PUT":
            puts_to_sell.append(ticker)

# Display Results
print("\n" + "="*80)
if len(results) > 0:
    print(f"🎯 FOUND {len(results)} HOURLY SETUPS FOR OPTION SELLING!")
    results_df = pd.DataFrame(results)
    
    try:
        from IPython.display import display, HTML
        display(HTML(results_df.to_html(index=False)))
    except:
        print(results_df.to_string(index=False))
        
    # COMMA SEPARATED TRADINGVIEW LISTS
    print("\n" + "="*80)
    print("📈 SELL CALL WATCHLIST (Bearish Rejection)")
    print("="*80)
    print(", ".join(calls_to_sell) if calls_to_sell else "None")
    
    print("\n" + "="*80)
    print("📉 SELL PUT WATCHLIST (Bullish Bounce)")
    print("="*80)
    print(", ".join(puts_to_sell) if puts_to_sell else "None")
    print("="*80)
    
else:
    print("No hourly setups found today rejecting historical untouched POCs.")
print("="*80)